# 🔧 Période 2 — Notebook 02 : Feature Engineering

**Objectif** : construire la matrice de features pour Naïve Bayes.

Features construites :
| Feature | Type | Source | Description |
|---------|------|--------|-------------|
| `gender_enc` | Binaire | users | 1=M, 0=F |
| `age` | Catégorielle | users | Tranche d'âge |
| `occupation` | Catégorielle | users | Code métier |
| `year` | Continue | movies | Année du film |
| `genre_*` | Binaire | movies | One-hot des genres |
| `user_avg_rating` | Continue | ratings | Sévérité moyenne de l'utilisateur |
| `user_n_ratings` | Continue | ratings | Activité de l'utilisateur |
| `movie_avg_rating` | Continue | ratings | Popularité du film |
| `movie_n_ratings` | Continue | ratings | Nombre de votes du film |

In [ ]:
import sys, json
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '..')
from training.data_loader import load_movies, load_users, load_ratings

movies  = load_movies()
users   = load_users()
ratings = load_ratings()

print('Données chargées :')
print(f'  movies  : {movies.shape}  colonnes = {list(movies.columns[:6])}...')
print(f'  users   : {users.shape}   colonnes = {list(users.columns)}')
print(f'  ratings : {ratings.shape} colonnes = {list(ratings.columns)}')

In [ ]:
# ── Étape 1 : Stats agrégées par utilisateur ────────────────────
user_stats = (ratings.groupby('UserID')
              .agg(
                  user_avg_rating  = ('Rating', 'mean'),
                  user_n_ratings   = ('Rating', 'count'),
                  user_std_rating  = ('Rating', 'std'),
              )
              .reset_index()
              .fillna(0))

print('Stats utilisateur (5 premières lignes) :')
print(user_stats.head())

In [ ]:
# ── Étape 2 : Stats agrégées par film ───────────────────────────
movie_stats = (ratings.groupby('MovieID')
               .agg(
                   movie_avg_rating = ('Rating', 'mean'),
                   movie_n_ratings  = ('Rating', 'count'),
               )
               .reset_index())

print('Stats film (5 premières lignes) :')
print(movie_stats.head())

In [ ]:
# ── Étape 3 : Fusion complète ────────────────────────────────────
from training.data_loader import build_feature_matrix, get_feature_columns

df = build_feature_matrix(ratings, users, movies)

# Ajout des stats
df = df.merge(user_stats,  on='UserID',  how='left')
df = df.merge(movie_stats, on='MovieID', how='left')
df = df.fillna(0)

print(f'Matrice finale : {df.shape}')
print(f'Taux de films aimés (Liked=1) : {df.Liked.mean():.1%}')
print(f'\nCo lonnes :')
for c in df.columns:
    print(f'  {c}')

In [ ]:
# ── Étape 4 : Définir et sauvegarder les feature columns ────────
STAT_FEATS = ['user_avg_rating','user_n_ratings','user_std_rating',
              'movie_avg_rating','movie_n_ratings']
BASE_FEATS = get_feature_columns(df)
# Retirer les stats de la liste de base (déjà ajoutées)
BASE_FEATS = [c for c in BASE_FEATS if c not in STAT_FEATS]
FEATURE_COLS = BASE_FEATS + STAT_FEATS

print(f'Total features : {len(FEATURE_COLS)}')
print(f'  Features utilisateur (continues)  : gender_enc, age, occupation')
print(f'  Features film (continues+binaires) : year + genres')
print(f'  Stats agrégées                     : {STAT_FEATS}')

# Sauvegarder
import json, os
os.makedirs('../data/processed', exist_ok=True)
with open('../data/processed/feature_columns.json','w') as f:
    json.dump(FEATURE_COLS, f, indent=2)
print('\n✅ feature_columns.json sauvegardé')

In [ ]:
# ── Étape 5 : Corrélation features / cible ───────────────────────
print('Corrélation des features avec Liked (top 10) :')
corr = df[FEATURE_COLS + ['Liked']].corr()['Liked'].drop('Liked').abs().sort_values(ascending=False)
for feat, val in corr.head(10).items():
    bar = '█' * int(val * 40)
    print(f'  {feat:25s} {bar}  {val:.4f}')

print('\n→ Les features les plus discriminantes sont les stats agrégées (movie_avg_rating, user_avg_rating)')

In [ ]:
# ── Étape 6 : Split train / test et sauvegarde ──────────────────
from sklearn.model_selection import train_test_split

X = df[FEATURE_COLS]
y = df['Liked']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Sauvegarder en CSV
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test .to_csv('../data/processed/X_test.csv',  index=False)
y_train.to_frame().to_csv('../data/processed/y_train.csv', index=False)
y_test .to_frame().to_csv('../data/processed/y_test.csv',  index=False)

print(f'✅ Split sauvegardé :')
print(f'  X_train : {X_train.shape}  ({y_train.mean():.1%} positifs)')
print(f'  X_test  : {X_test.shape}   ({y_test.mean():.1%} positifs)')
print(f'\n→ Passer au notebook 03_training_experiment.ipynb')